# ME 313 · Lab 3.1 — A Surrogate for the Fin-Efficiency Chart (Module 3)
**Goal:** replace the fin-efficiency chart with a trained model, then use it to *size a fin* — and check it against the exact formula $\eta_f=\tanh(mL_c)/(mL_c)$.

Because the ground truth is analytic, this is a clean way to see what a surrogate buys you. **Free Colab, no GPU needed.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


### 1. Generate the efficiency curve and train a surrogate


In [ ]:
mLc = np.linspace(0.1, 4.0, 800)
eta = np.tanh(mLc)/mLc                       # exact fin efficiency
sur = make_pipeline(StandardScaler(),
                    MLPRegressor(hidden_layer_sizes=(64,64), activation='tanh',
                                 max_iter=12000, random_state=0))
sur.fit(mLc.reshape(-1,1), eta)              # TODO: fit the surrogate
err = np.max(np.abs(sur.predict(mLc.reshape(-1,1)) - eta))
print(f'surrogate max |error| vs analytical = {err:.4f}')


In [ ]:
plt.figure(figsize=(5.5,4))
plt.plot(mLc, eta, 'k', lw=2, label='analytical  tanh(mLc)/mLc')
plt.plot(mLc, sur.predict(mLc.reshape(-1,1)), 'r--', lw=1.6, label='surrogate')
plt.xlabel('mL_c'); plt.ylabel('fin efficiency'); plt.legend(); plt.title('Lab 3.1 — surrogate vs. chart')
plt.tight_layout(); plt.show()


### 2. Use the surrogate to size a fin
Aluminium straight fin: $k=240$, $h=40$, thickness $t=3$ mm, width $w=0.1$ m, base excess $\theta_b=150$ K.
Find the length $L$ that delivers a target $Q_f = 25$ W, using the surrogate for $\eta_f$, and compare with the exact formula.


In [ ]:
k,h,t,w,thb = 240.,40.,0.003,0.1,150.
m = np.sqrt(2*h/(k*t))
def Qf_of_L(L, eta_fn):
    Lc = L + t/2
    eta = float(np.asarray(eta_fn(np.array([[m*Lc]]))).ravel()[0])
    Af  = 2*w*Lc
    return eta*h*Af*thb
Ls = np.linspace(0.005, 0.06, 400)
Q_sur = np.array([Qf_of_L(L, sur.predict) for L in Ls])
Q_exact = np.array([Qf_of_L(L, lambda a: np.tanh(a)/a) for L in Ls])
L_sur   = Ls[np.argmin(np.abs(Q_sur   - 25))]
L_exact = Ls[np.argmin(np.abs(Q_exact - 25))]
print(f'length for Q_f=25 W:  surrogate={1e3*L_sur:.1f} mm   exact={1e3*L_exact:.1f} mm')


### 3. Reflect
- The surrogate reproduces the chart and gives essentially the same sizing as the exact formula.
- **Where it helps:** instant evaluation inside a design optimizer that calls $\eta_f$ thousands of times.
- **Verification habit:** the surrogate is only valid on the range it saw ($0.1\le mL_c\le 4$). Outside that, fall back to the formula.
